# 🧭 AI Career Coach — Complete Kaggle Notebook

**100% free · no API keys · local LLM · local embeddings · local FAISS · RAG · chains · Pydantic structured output · Streamlit**

Before *Run All*: (1) add your `rag_knowledge_base.json` as a Kaggle input, (2) turn **GPU ON**, (3) turn **Internet ON** (only needed once, to download the open-source model weights and the free tunnel binary — the app itself calls no external API).

Cells 3–12 write the project's modules into `/kaggle/working/career_coach/` and immediately use them, so the Streamlit app runs the **exact same code** that the tests run. The last cell starts Streamlit and prints the public URL.

## Table of contents

1. **Step 1** — Install dependencies (all free / open-source)
2. **Step 2** — Imports, environment detection and module-writer helper
3. **Step 3** — Automatically locate the RAG file in /kaggle/input/
4. **Step 4** — Load the RAG knowledge base
5. **Step 5** — Initialise the LOCAL embedding model (BAAI/bge-small-en-v1.5)
6. **Step 6** — Build the FAISS vector store (local) + role/skill index + metadata
7. **Step 7** — Load the LOCAL LLM (Qwen2.5 instruct, open-source, runs inside Kaggle)
8. **Step 8** — Pydantic schemas (structured output)
9. **Step 9** — Prompt engineering (one ChatPromptTemplate per chain)
10. **Step 10** — Retriever (task-specific semantic retrieval with metadata filtering)
11. **Step 11** — Chains (prompt -> local LLM -> output parser -> Pydantic) + skill matching
12. **Step 12** — Career Coach pipeline (+ local PDF CV parser)
13. **Step 13** — Automated tests (no external API needed)
14. **Step 14** — Create the Streamlit application (app.py + theme)
15. **Final step** — Launch Streamlit + free Cloudflare tunnel (no account, no API key). Keep this cell running.

**Pipeline:** CV (PDF) → Chain 1 CV analysis → Chain 4 career ranking → Chain 2 requirements → Chain 3 skill gaps → Chain 5 roadmap → Chain 6 final report → Streamlit UI.

---
## Step 1 · Install dependencies (all free / open-source)

Installs every open-source dependency used by the project.

In [ ]:
# CELL 1 — Install dependencies (all free / open-source)
!pip install -q "transformers>=4.44" accelerate sentence-transformers faiss-cpu pymupdf pypdf streamlit langchain-core "pydantic>=2"
print("✅ Dependencies installed")

---
## Step 2 · Imports, environment detection and module-writer helper

Imports, GPU/CPU detection and the `write_module` helper that writes each project module to disk and imports it.

In [ ]:
# CELL 2 — Imports, environment detection and module-writer helper
import os, sys, re, gc, json, time, importlib, platform
from pathlib import Path
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
import torch

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
PKG = WORK / "career_coach"
PKG.mkdir(exist_ok=True)
(PKG / "__init__.py").write_text("")
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

def write_module(name, code):
    """Write a module of the project to disk and import (or reload) it."""
    (PKG / f"{name}.py").write_text(code, encoding="utf-8")
    full = f"career_coach.{name}"
    if full in sys.modules:
        return importlib.reload(sys.modules[full])
    return importlib.import_module(full)

HAS_GPU = torch.cuda.is_available()
print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("Device      :", "CUDA GPU -> " + torch.cuda.get_device_name(0) if HAS_GPU else "CPU (fallback, slower)")
if HAS_GPU:
    print("GPU memory  : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
print("Working dir :", WORK)

---
## Step 3 · Automatically locate the RAG file in /kaggle/input/

Writes the `rag_loader` module and locates `rag_knowledge_base.json` in `/kaggle/input/`.

In [ ]:
# CELL 3 — Automatically locate the RAG file in /kaggle/input/
write_module("rag_loader", r'''
import json
import os
from collections import Counter

RAG_FILENAME = "rag_knowledge_base.json"


class RAGError(Exception):
    """A human-readable problem with the knowledge-base file."""


def find_rag_file(filename=RAG_FILENAME, roots=("/kaggle/input", "/kaggle/working", ".")):
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, files in os.walk(root):
            for f in files:
                if f.lower() == filename.lower():
                    return os.path.join(dirpath, f)
    raise RAGError(
        "rag_knowledge_base.json was not found in /kaggle/input/. "
        "Upload it to Kaggle (Add Input -> Upload -> new dataset containing rag_knowledge_base.json), "
        "attach it to this notebook and run all cells again."
    )


def _s(v):
    if v is None:
        return ""
    if isinstance(v, (list, tuple)):
        return ", ".join(str(x) for x in v)
    return str(v).strip()


def _l(v):
    if v is None:
        return []
    if isinstance(v, (list, tuple)):
        return [str(x).strip() for x in v if str(x).strip()]
    return [x.strip() for x in str(v).replace(";", ",").split(",") if x.strip()]


def load_records(path):
    """Load + validate the knowledge base. Returns a list of normalised records."""
    try:
        with open(path, "r", encoding="utf-8") as fh:
            data = json.load(fh)
    except json.JSONDecodeError as e:
        raise RAGError(f"The RAG file is not valid JSON (problem near line {e.lineno}). Please re-export it.")
    except OSError as e:
        raise RAGError(f"The RAG file could not be read: {e}")

    if isinstance(data, dict):
        for key in ("records", "data", "documents", "items"):
            if isinstance(data.get(key), list):
                data = data[key]
                break
        else:
            data = [data]
    if not isinstance(data, list):
        raise RAGError("The RAG file must contain a JSON list of records with 'id', 'text' and 'metadata'.")

    records, seen = [], set()
    for i, item in enumerate(data):
        if not isinstance(item, dict):
            continue
        text = _s(item.get("text"))
        if len(text) < 5:
            continue
        md = item.get("metadata")
        if not isinstance(md, dict):
            md = {}
        rid = _s(item.get("id")) or f"rec_{i}"
        if rid in seen:
            rid = f"{rid}_{i}"
        seen.add(rid)
        records.append({
            "id": rid,
            "text": text,
            "metadata": {
                "domain": _s(md.get("domain")),
                "role": _s(md.get("role")),
                "skill": _s(md.get("skill")),
                "category": _s(md.get("category")),
                "level": _s(md.get("level")),
                "priority": _s(md.get("priority")),
                "related_roles": _l(md.get("related_roles")),
                "source_type": _s(md.get("source_type")),
            },
        })
    if not records:
        raise RAGError("The RAG file is empty or contains no usable records (each record needs a non-empty 'text').")
    return records


def summarize_records(records):
    def count(k):
        return Counter(r["metadata"][k] for r in records if r["metadata"][k])
    return {
        "records": len(records),
        "distinct_roles": len(count("role")),
        "distinct_domains": len(count("domain")),
        "top_categories": dict(count("category").most_common(8)),
    }
''')

from career_coach.rag_loader import RAGError, find_rag_file, load_records, summarize_records

RAG_PATH, RAG_ERROR = None, None
try:
    RAG_PATH = find_rag_file()
    print("✅ RAG file found:", RAG_PATH)
except RAGError as e:
    RAG_ERROR = str(e)
    print("❌ " + RAG_ERROR)

---
## Step 4 · Load the RAG knowledge base

Loads and validates the knowledge base.

In [ ]:
# CELL 4 — Load the RAG knowledge base
RECORDS = None
if RAG_PATH:
    try:
        RECORDS = load_records(RAG_PATH)
        print("✅ Knowledge base loaded:", summarize_records(RECORDS))
        print("Example record:", json.dumps(RECORDS[0], indent=1)[:600])
    except RAGError as e:
        RAG_ERROR = str(e)
        print("❌ " + RAG_ERROR)
else:
    print("⏭️  Skipped — no RAG file. The Streamlit app will show the upload instructions.")

---
## Step 5 · Initialise the LOCAL embedding model (BAAI/bge-small-en-v1.5)

Writes the `embeddings` module and loads the local embedding model (BAAI/bge-small-en-v1.5).

In [ ]:
# CELL 5 — Initialise the LOCAL embedding model (BAAI/bge-small-en-v1.5)
write_module("embeddings", r'''
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "


class LocalEmbedder:
    """Runs fully locally (GPU if available). Vectors are L2-normalised, so inner product = cosine."""

    def __init__(self, model_name=EMBED_MODEL):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(model_name, device=self.device)
        if self.device == "cuda":
            self.model.half()
        self.dim = self.model.get_sentence_embedding_dimension()
        self.model_name = model_name

    def embed_documents(self, texts, batch_size=64):
        if not texts:
            return np.zeros((0, self.dim), dtype="float32")
        v = self.model.encode(list(texts), batch_size=batch_size, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False)
        return v.astype("float32")

    def embed_query(self, text):
        v = self.model.encode([QUERY_PREFIX + text], normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False)
        return v.astype("float32")
''')
from career_coach.embeddings import LocalEmbedder

embedder = LocalEmbedder()
_v = embedder.embed_documents(["python developer", "machine learning engineer"])
print(f"✅ Embedding model ready: {embedder.model_name} | device={embedder.device} | dim={embedder.dim} | test shape={_v.shape}")

---
## Step 6 · Build the FAISS vector store (local) + role/skill index + metadata

Writes the `vectorstore` module and builds the local FAISS index plus the role/skill index.

In [ ]:
# CELL 6 — Build the FAISS vector store (local) + role/skill index + metadata
write_module("vectorstore", r'''
import re
from collections import defaultdict

import faiss
import numpy as np

GENERIC_ROLES = {"", "general", "all", "any", "n/a", "na", "none", "common", "multiple", "various"}


def norm(s):
    s = re.sub(r"[^a-z0-9+#.]+", " ", str(s).lower())
    s = re.sub(r"\.+(?=\s|$)", "", s)
    return re.sub(r"\s+", " ", s).strip()


def split_skills(s):
    return [x.strip() for x in re.split(r"[;,|]", s or "") if x.strip()]


def priority_weight(p):
    p = (p or "").lower()
    if any(k in p for k in ("critical", "high", "essential", "must", "core", "required")):
        return 3
    if any(k in p for k in ("low", "optional", "nice", "bonus")):
        return 1
    return 2


class FaissStore:
    """RAG JSON -> records -> embeddings -> FAISS index (+ metadata kept beside the vectors)."""

    def __init__(self, records, embedder):
        self.records = records
        self.embedder = embedder
        self.by_id = {r["id"]: r for r in records}
        vecs = embedder.embed_documents([self._doc_text(r) for r in records])
        self.index = faiss.IndexFlatIP(vecs.shape[1])      # exact cosine search, fully local
        self.index.add(vecs)
        self._skill_cache = {}
        self._role_vecs = None
        self._build_roles()

    @staticmethod
    def _doc_text(r):
        m = r["metadata"]
        head = " | ".join(x for x in (m["role"], m["skill"], m["category"]) if x)
        return f"{head}: {r['text']}" if head else r["text"]

    # ---------- role / skill index derived from the metadata ----------
    def _build_roles(self):
        self.role_names = {}
        self.role_skills = defaultdict(dict)
        self._rec_roles = []
        for r in self.records:
            m = r["metadata"]
            primary = norm(m["role"])
            primary = None if primary in GENERIC_ROLES else primary
            if primary:
                self.role_names.setdefault(primary, m["role"].strip())
            member = {primary} if primary else set()
            related = []
            for rr in m["related_roles"]:
                k = norm(rr)
                if k and k not in GENERIC_ROLES:
                    self.role_names.setdefault(k, rr.strip())
                    related.append(k)
                    member.add(k)
            self._rec_roles.append(member)
            owners = [primary] if primary else related
            w = priority_weight(m["priority"])
            for sk in split_skills(m["skill"]):
                sk_key = norm(sk)
                if not sk_key:
                    continue
                for o in owners:
                    cur = self.role_skills[o].get(sk_key)
                    if cur is None:
                        self.role_skills[o][sk_key] = {"name": sk, "weight": w}
                    elif w > cur["weight"]:
                        cur["weight"] = w
        self.vocab = {}
        for d in self.role_skills.values():
            for k, v in d.items():
                self.vocab.setdefault(k, v["name"])
        # roles whose records carry no 'skill' metadata: scan their text for known skills
        for rk in list(self.role_names):
            if len(self.role_skills.get(rk, {})) >= 3 or not self.vocab:
                continue
            for i, r in enumerate(self.records):
                if rk not in self._rec_roles[i]:
                    continue
                t = " " + norm(r["text"]) + " "
                for vk, vn in self.vocab.items():
                    if len(vk) > 3 and (" " + vk + " ") in t:
                        self.role_skills[rk].setdefault(vk, {"name": vn, "weight": 2})

    def rankable_roles(self):
        return [k for k, v in self.role_skills.items() if len(v) >= 2]

    def all_skills(self):
        return list(self.vocab.values())

    def resolve_role(self, name):
        k = norm(name)
        if not k or not self.role_names:
            return None
        if k in self.role_names:
            return k
        cands = [rk for rk in self.role_names if k in rk or rk in k]
        if cands:
            return min(cands, key=lambda rk: abs(len(rk) - len(k)))
        keys = list(self.role_names)
        if self._role_vecs is None:
            self._role_vecs = self.embedder.embed_documents([self.role_names[x] for x in keys])
        q = self.embedder.embed_documents([name])[0]
        sims = self._role_vecs @ q
        j = int(np.argmax(sims))
        return keys[j] if sims[j] >= 0.80 else None

    def embed_skills(self, names):
        keys = [norm(n) for n in names]
        todo = {}
        for k, n in zip(keys, names):
            if k not in self._skill_cache:
                todo.setdefault(k, n)
        if todo:
            vecs = self.embedder.embed_documents(list(todo.values()))
            for k, v in zip(todo, vecs):
                self._skill_cache[k] = v
        if not keys:
            return np.zeros((0, self.embedder.dim), dtype="float32")
        return np.stack([self._skill_cache[k] for k in keys])

    # ---------- semantic search with metadata filtering ----------
    def search(self, query, k=5, role=None, categories=None, exclude_ids=()):
        n = self.index.ntotal
        fetch = n if n <= 3000 else min(n, max(k * 25, 300))
        scores, idx = self.index.search(self.embedder.embed_query(query), fetch)
        out = []
        for s, i in zip(scores[0], idx[0]):
            if i < 0:
                continue
            r = self.records[i]
            if r["id"] in exclude_ids:
                continue
            if role and role not in self._rec_roles[i]:
                continue
            if categories:
                hay = (r["metadata"]["category"] + " " + r["metadata"]["source_type"]).lower()
                if not any(c in hay for c in categories):
                    continue
            out.append({**r, "score": float(s)})
            if len(out) >= k:
                break
        return out
''')
from career_coach.vectorstore import FaissStore

store = None
if RECORDS:
    store = FaissStore(RECORDS, embedder)
    print(f"✅ FAISS index built: {store.index.ntotal} vectors | roles={len(store.role_names)} | rankable roles={len(store.rankable_roles())} | known skills={len(store.vocab)}")
    if len(store.rankable_roles()) == 0:
        print("⚠️  No role has skill metadata. Fill the 'role' and 'skill' metadata fields for career matching to work.")
else:
    print("⏭️  Skipped — no knowledge base loaded.")

---
## Step 7 · Load the LOCAL LLM (Qwen2.5 instruct, open-source, runs inside Kaggle)

Writes the `llm` module and loads the local Qwen2.5 instruct model.

In [ ]:
# CELL 7 — Load the LOCAL LLM (Qwen2.5 instruct, open-source, runs inside Kaggle)
write_module("llm", r'''
import gc
import threading

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen2.5-3B-Instruct"      # ~6.5 GB in fp16: fits a 16 GB T4/P100 with plenty of headroom
CPU_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"    # smaller model for CPU-only fallback


class LocalLLM:
    def __init__(self):
        self.lock = threading.Lock()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        try:
            self._load(GPU_MODEL if self.device == "cuda" else CPU_MODEL)
        except Exception:
            if self.device != "cuda":
                raise
            self.free()
            self.device = "cpu"          # e.g. GPU out-of-memory -> fall back to the small CPU model
            self._load(CPU_MODEL)

    def _load(self, name):
        self.model_name = name
        self.tok = AutoTokenizer.from_pretrained(name)
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype, low_cpu_mem_usage=True)
        self.model.to(self.device)
        self.model.eval()

    def generate(self, messages, max_new_tokens=800):
        """messages = [{'role': 'system'|'user'|'assistant', 'content': str}, ...] -> generated text."""
        text = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tok(text, return_tensors="pt", truncation=True, max_length=7000).to(self.device)
        with self.lock, torch.inference_mode():
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      repetition_penalty=1.05, pad_token_id=self.tok.eos_token_id)
        return self.tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    def free(self):
        for attr in ("model", "tok"):
            if hasattr(self, attr):
                delattr(self, attr)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
''')
from career_coach.llm import LocalLLM

llm, LLM_ERROR = None, None
try:
    t0 = time.time()
    llm = LocalLLM()
    print(f"✅ Local LLM ready: {llm.model_name} on {llm.device} ({time.time()-t0:.0f}s)")
except Exception as e:
    LLM_ERROR = f"{type(e).__name__}: {e}"
    print("❌ Could not load the local LLM:", LLM_ERROR, "\n   (Is Internet ON for the first download? Is the GPU enabled?)")

---
## Step 8 · Pydantic schemas (structured output)

Writes the `schemas` module: lenient Pydantic models for structured output.

In [ ]:
# CELL 8 — Pydantic schemas (structured output)
write_module("schemas", r'''
import re

from pydantic import BaseModel, ConfigDict, Field, model_validator

STAGES = ["Foundation", "Core Skills", "Advanced Skills", "Projects", "Deployment", "Job Readiness"]


def _to_text(v):
    if isinstance(v, dict):
        return "; ".join(f"{k}: {_to_text(x)}" for k, x in v.items() if x not in (None, ""))
    if isinstance(v, (list, tuple)):
        return ", ".join(_to_text(x) for x in v)
    return str(v).strip()


class Lenient(BaseModel):
    """Base class: tolerates the usual small-LLM mistakes (null, string instead of list, '3 years' as number...)."""
    model_config = ConfigDict(extra="ignore")

    @model_validator(mode="before")
    @classmethod
    def _coerce(cls, data):
        if not isinstance(data, dict):
            return data
        out = {}
        for key, val in data.items():
            if val is None:
                continue
            field = cls.model_fields.get(key)
            ann = field.annotation if field else None
            if ann == list[str]:
                if isinstance(val, list):
                    val = [_to_text(x) for x in val if x not in (None, "")]
                else:
                    val = [_to_text(val)] if str(val).strip() else []
            elif ann is str and not isinstance(val, str):
                val = _to_text(val)
            elif ann is float and not isinstance(val, (int, float)):
                m = re.search(r"\d+(?:\.\d+)?", str(val))
                val = float(m.group()) if m else 0.0
            out[key] = val
        return out


# ---------------- Chain 1 output ----------------
class CVProfile(Lenient):
    name: str = "Unknown"
    education: list[str] = []
    experience: list[str] = []
    experience_level: str = "Entry"
    years_of_experience: float = 0.0
    skills: list[str] = []
    programming_languages: list[str] = []
    frameworks: list[str] = []
    libraries: list[str] = []
    databases: list[str] = []
    cloud_technologies: list[str] = []
    tools: list[str] = []
    projects: list[str] = []
    certifications: list[str] = []
    soft_skills: list[str] = []

    def all_skills(self):
        seen, out = set(), []
        for grp in (self.skills, self.programming_languages, self.frameworks, self.libraries,
                    self.databases, self.cloud_technologies, self.tools):
            for s in grp:
                for part in re.split(r"[;,|]", s):
                    p = part.strip()
                    if p and p.lower() not in seen:
                        seen.add(p.lower())
                        out.append(p)
        return out


# ---------------- Chain 3 ----------------
class SkillMatch(Lenient):
    skill: str = ""
    status: str = "Missing"          # Strong Match | Match | Partial Match | Missing
    priority: str = "Medium"
    matched_with: str = ""


class SkillGap(Lenient):
    skill: str = ""
    status: str = "Missing"
    priority: str = "Medium"         # High | Medium | Low (taken from the knowledge-base metadata)
    reason: str = ""
    prerequisites: list[str] = []
    learning_order: int = 0
    evidence_ids: list[str] = []


class GapDetail(Lenient):
    skill: str = ""
    reason: str = ""
    prerequisites: list[str] = []


class GapEnrichment(Lenient):          # what the LLM returns for Chain 3
    gaps: list[GapDetail] = []


# ---------------- Chain 4 ----------------
class RoleRecommendation(Lenient):
    role: str = ""
    score: float = 0.0               # "Profile Match Score" - NOT a hiring probability
    matched_skills: list[str] = []
    missing_skills: list[str] = []
    rationale: str = ""


# ---------------- Chain 5 ----------------
class RoadmapStep(Lenient):
    stage: str = "Core Skills"
    skill: str = ""
    why_it_matters: str = ""
    priority: str = "Medium"
    prerequisites: list[str] = []
    suggested_practice: str = ""
    suggested_project: str = ""


class RoadmapOutput(Lenient):
    steps: list[RoadmapStep] = []


# ---------------- Chain 6 ----------------
class ProjectIdea(Lenient):
    title: str = ""
    description: str = ""
    skills_covered: list[str] = []


class Narrative(Lenient):
    overall_assessment: str = ""
    projects: list[ProjectIdea] = []
    improvement_plan: list[str] = []
    job_readiness: list[str] = []


class EvidenceItem(Lenient):
    id: str = ""
    role: str = ""
    skill: str = ""
    category: str = ""
    source_type: str = ""
    snippet: str = ""
    used_for: str = ""


class CareerReport(Lenient):
    target_role: str = ""
    auto_recommended: bool = False
    profile_match_score: float = 0.0
    score_note: str = ""
    experience_level: str = "Entry"
    overall_assessment: str = ""
    profile: CVProfile = Field(default_factory=CVProfile)
    current_skills: list[str] = []
    strong_skills: list[str] = []
    partial_skills: list[str] = []
    missing_skills: list[str] = []
    skill_matches: list[SkillMatch] = []
    skill_gaps: list[SkillGap] = []
    recommendations: list[RoleRecommendation] = []
    roadmap: list[RoadmapStep] = []
    projects: list[ProjectIdea] = []
    improvement_plan: list[str] = []
    job_readiness: list[str] = []
    evidence: list[EvidenceItem] = []
    pipeline_status: dict = {}
    warnings: list[str] = []
''')
from career_coach.schemas import *

print("✅ Schemas defined:", [c.__name__ for c in (CVProfile, SkillGap, RoleRecommendation, RoadmapStep, CareerReport)])

---
## Step 9 · Prompt engineering (one ChatPromptTemplate per chain)

Writes the `prompts` module: one `ChatPromptTemplate` per chain.

In [ ]:
# CELL 9 — Prompt engineering (one ChatPromptTemplate per chain)
write_module("prompts", r'''
import json
from typing import get_args, get_origin

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel

SYSTEM = (
    "You are an expert career coach and technical recruiter. "
    "Reply with ONE valid JSON object and nothing else: no markdown, no comments, no explanations. "
    "Be concrete, concise and faithful to the provided data. Never invent facts about the candidate."
)


def _example(ann):
    if get_origin(ann) is list:
        return [_example(get_args(ann)[0])]
    if isinstance(ann, type) and issubclass(ann, BaseModel):
        return skeleton(ann)
    if ann in (int, float):
        return 0
    return "string"


def skeleton(model):
    return {name: _example(f.annotation) for name, f in model.model_fields.items()}


def format_hint(model):
    """Output-format instruction: a JSON skeleton generated from the Pydantic schema (small models follow it well)."""
    return "Return ONLY a JSON object with exactly this structure (replace the placeholders):\n" + json.dumps(skeleton(model), indent=1)


CV_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human",
     "Extract structured information from the CV text below.\n"
     "Rules: use ONLY information explicitly present in the CV; use empty lists when something is missing; "
     "list each skill as a short name (e.g. 'Python', 'PyTorch', 'PostgreSQL'); put every technical skill also in 'skills'. "
     "experience_level must be one of: Student, Entry, Junior, Mid, Senior. "
     "Each experience entry is one short line like 'Data Analyst - Acme (2021-2023)'.\n"
     "{format_instructions}\n\nCV TEXT:\n{cv_text}"),
])

GAP_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human",
     "Target role: {role}\nCandidate level: {level}\n"
     "Skills the candidate lacks or only partly has (most important first):\n{missing}\n\n"
     "Knowledge-base evidence about the role and skills:\n{context}\n\n"
     "For EACH skill above write: a one-sentence reason why it matters for this role (use the evidence when possible) "
     "and its prerequisites (skills to learn first; empty list if none). Use exactly the skill names given.\n"
     "{format_instructions}"),
])

ROADMAP_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human",
     "Create a PERSONALISED learning roadmap for this candidate.\n"
     "Target role: {role}\nExperience level: {level}\n"
     "Skills the candidate already has: {strengths}\n"
     "Projects the candidate already built: {cv_projects}\n"
     "Skill gaps in recommended learning order (priority | prerequisites):\n{gaps}\n\n"
     "Knowledge-base evidence (learning order, dependencies, projects):\n{context}\n\n"
     "Rules: 6 to 9 steps, ordered by learning order. Each step has a stage chosen from: "
     "Foundation, Core Skills, Advanced Skills, Projects, Deployment, Job Readiness. "
     "Build on what the candidate already knows (mention it in why_it_matters), do not teach skills they already have, "
     "and make suggested_practice and suggested_project specific and doable.\n"
     "{format_instructions}"),
])

REPORT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human",
     "Target role: {role}\nCandidate experience level: {level}\n"
     "Profile Match Score (internal matching score, NOT a hiring probability): {score}/100\n"
     "Candidate strengths: {strengths}\nProjects already built: {cv_projects}\n"
     "Priority gaps to close: {gaps}\nRoadmap focus: {roadmap_focus}\n\n"
     "Knowledge-base evidence:\n{context}\n\n"
     "Write the final coaching summary:\n"
     "- overall_assessment: 3 to 4 honest, specific sentences; call the number a profile match score.\n"
     "- projects: exactly 3 portfolio projects that close the priority gaps (title, 2-sentence description, skills_covered).\n"
     "- improvement_plan: 5 to 7 concrete next steps ordered by priority, including CV and portfolio improvements.\n"
     "- job_readiness: 4 to 5 short statements about what is ready and what is not yet ready for applying.\n"
     "Never state hiring probabilities, salaries or statistics.\n"
     "{format_instructions}"),
])
''')
print("✅ Prompts defined: CV, GAP, ROADMAP, REPORT")

---
## Step 10 · Retriever (task-specific semantic retrieval with metadata filtering)

Writes the `retriever` module: task-specific retrieval with metadata filtering.

In [ ]:
# CELL 10 — Retriever (task-specific semantic retrieval with metadata filtering)
write_module("retriever", r'''
from .vectorstore import norm

CAT_REQ = ["requirement", "role", "job", "skill", "responsib"]
CAT_ROAD = ["roadmap", "learning", "path", "project", "depend", "prereq", "order"]


def format_context(docs, max_chars=380):
    if not docs:
        return "(no relevant knowledge-base entries found)"
    lines = []
    for d in docs:
        m = d["metadata"]
        tag = " / ".join(x for x in (m["role"], m["skill"], m["category"]) if x)
        lines.append(f"[{d['id']}] ({tag}) {d['text'][:max_chars].strip()}")
    return "\n".join(lines)


class CareerRetriever:
    """Retrieves ONLY the few knowledge-base entries needed by each task (never the whole dataset)."""

    def __init__(self, store):
        self.store = store

    def _search(self, query, k, role=None, categories=None):
        res = self.store.search(query, k=k, role=role, categories=categories)
        if len(res) < 2 and categories:                 # relax the category filter first
            res = self.store.search(query, k=k, role=role)
        if len(res) < 2 and role:                       # then the role filter
            res = self.store.search(query, k=k, categories=categories) or self.store.search(query, k=k)
        return res

    def requirements(self, role_key, role_name, k=6):
        return self._search(f"{role_name} job requirements required skills responsibilities", k, role_key, CAT_REQ)

    def for_gaps(self, role_key, role_name, skills, per_skill=1, cap=8):
        docs, seen = [], set()
        for s in skills:
            got = self._search(f"{s} skill for {role_name}", per_skill + 2, role_key)
            got.sort(key=lambda d: 0 if norm(s) in norm(d["metadata"]["skill"]) else 1)
            for d in got[:per_skill]:
                if d["id"] not in seen:
                    seen.add(d["id"])
                    docs.append(d)
        return docs[:cap]

    def for_roadmap(self, role_key, role_name, skills, cap=9):
        docs, seen = [], set()
        for d in self._search(f"{role_name} learning roadmap order dependencies recommended projects", 5, role_key, CAT_ROAD):
            if d["id"] not in seen:
                seen.add(d["id"])
                docs.append(d)
        for s in skills[:4]:
            for d in self._search(f"prerequisites and learning path for {s}", 1, role_key, CAT_ROAD):
                if d["id"] not in seen:
                    seen.add(d["id"])
                    docs.append(d)
        return docs[:cap]
''')
from career_coach.retriever import CareerRetriever, format_context

retriever = CareerRetriever(store) if store else None
if retriever and store.rankable_roles():
    _rk = store.rankable_roles()[0]
    _docs = retriever.requirements(_rk, store.role_names[_rk])
    print(f"✅ Retriever ready. Sample query for role '{store.role_names[_rk]}' returned {len(_docs)} entries:")
    print(format_context(_docs[:2], 160))
else:
    print("✅ Retriever module written (waiting for a knowledge base).")

---
## Step 11 · Chains (prompt -> local LLM -> output parser -> Pydantic) + skill matching

Writes the `matching` and `chains` modules: skill matching and the six chains (prompt → LLM → parser → Pydantic, with repair and fallback).

In [ ]:
# CELL 11 — Chains (prompt -> local LLM -> output parser -> Pydantic) + skill matching
write_module("matching", r'''
import re

import numpy as np

from .vectorstore import norm

SEM_MATCH = 0.86
SEM_PARTIAL = 0.74
FACTOR = {"Strong Match": 1.0, "Match": 0.8, "Partial Match": 0.4, "Missing": 0.0}


class SkillMatcher:
    """Compares the candidate's skills with required skills: exact text -> containment -> embedding similarity."""

    def __init__(self, store):
        self.store = store

    def prepare(self, user_skills, cv_text):
        user = [s for s in dict.fromkeys(x.strip() for x in user_skills) if s]
        return {
            "user": user,
            "user_norm": {norm(s): s for s in user},
            "uvecs": self.store.embed_skills(user),
            "cv_text": cv_text,
            "cv_n": " " + norm(cv_text) + " ",
        }

    @staticmethod
    def in_text(name, key, ctx):
        if len(key) <= 3:       # short names (R, C, Go...) need a case-sensitive whole-word match
            return re.search(r"(?<![A-Za-z0-9+#])" + re.escape(name) + r"(?![A-Za-z0-9+#])", ctx["cv_text"]) is not None
        return (" " + key + " ") in ctx["cv_n"]

    @staticmethod
    def _contains(a, b):
        return len(a) >= 3 and len(b) >= 3 and ((" " + b + " ") in (" " + a + " ") or (" " + a + " ") in (" " + b + " "))

    def classify(self, required, ctx):
        names = [v["name"] for v in required.values()]
        rvecs = self.store.embed_skills(names)
        sims = rvecs @ ctx["uvecs"].T if len(ctx["user"]) else None
        out = []
        for idx, (key, info) in enumerate(required.items()):
            status, sim, via = "Missing", 0.0, ""
            if key in ctx["user_norm"]:
                status, sim, via = "Strong Match", 1.0, ctx["user_norm"][key]
            elif self.in_text(info["name"], key, ctx):
                status, sim, via = "Strong Match", 1.0, "mentioned in CV"
            else:
                contain = next((u for nk, u in ctx["user_norm"].items() if self._contains(nk, key)), None)
                if contain:
                    status, sim, via = "Match", 0.9, contain
                elif sims is not None:
                    j = int(np.argmax(sims[idx]))
                    sim = float(sims[idx][j])
                    if sim >= SEM_MATCH:
                        status, via = "Match", ctx["user"][j]
                    elif sim >= SEM_PARTIAL:
                        status, via = "Partial Match", ctx["user"][j]
            out.append({"skill": info["name"], "key": key, "weight": info["weight"], "status": status,
                        "similarity": round(sim, 3), "matched_with": via})
        return out

    @staticmethod
    def score(matches):
        total = sum(m["weight"] for m in matches)
        if not total:
            return 0.0
        return round(100.0 * sum(m["weight"] * FACTOR[m["status"]] for m in matches) / total, 1)

    def rank_roles(self, ctx):
        ranked = []
        for key in self.store.rankable_roles():
            ms = self.classify(self.store.role_skills[key], ctx)
            ranked.append({"key": key, "role": self.store.role_names[key], "score": self.score(ms), "matches": ms})
        ranked.sort(key=lambda r: r["score"], reverse=True)
        return ranked
''')

write_module("chains", r'''
import json
import re

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableLambda

from . import prompts as P
from .retriever import format_context
from .schemas import *
from .vectorstore import norm

PRIORITY = {3: "High", 2: "Medium", 1: "Low"}


def clean_json(text):
    """Strip code fences / chatter around the JSON object and drop trailing commas."""
    t = text.strip()
    t = re.sub(r"^```(?:json)?", "", t).strip()
    t = re.sub(r"```$", "", t).strip()
    s, e = t.find("{"), t.rfind("}")
    if s != -1 and e > s:
        t = t[s:e + 1]
    return re.sub(r",\s*([}\]])", r"\1", t)


def llm_runnable(llm, max_new_tokens):
    roles = {"human": "user", "ai": "assistant", "system": "system"}

    def _call(prompt_value):
        msgs = [{"role": roles.get(m.type, "user"), "content": m.content} for m in prompt_value.to_messages()]
        return llm.generate(msgs, max_new_tokens)
    return RunnableLambda(_call)


class StructuredChain:
    """prompt | local LLM | clean | PydanticOutputParser  with  retry-by-repair  and a safe fallback."""

    def __init__(self, name, prompt, llm, model, max_new_tokens, fallback):
        self.name, self.llm, self.model, self.fallback = name, llm, model, fallback
        self.max_new_tokens = max_new_tokens
        self.parser = PydanticOutputParser(pydantic_object=model)      # OUTPUT PARSER
        self.last_raw = ""

        def capture(text):
            self.last_raw = text
            return text

        self.chain = (prompt.partial(format_instructions=P.format_hint(model))       # PROMPT
                      | llm_runnable(llm, max_new_tokens)                              # LOCAL LLM
                      | RunnableLambda(capture)
                      | RunnableLambda(clean_json)
                      | self.parser)                                                   # -> Pydantic object

    def _repair(self, bad, error):
        msgs = [
            {"role": "system", "content": P.SYSTEM},
            {"role": "user", "content": "The text below should be ONE valid JSON object.\n" + P.format_hint(self.model)
             + f"\n\nPROBLEM: {error[:300]}\n\nTEXT:\n{bad[:6000]}\n\nReturn ONLY the corrected JSON object. Do not add new facts."},
        ]
        return self.llm.generate(msgs, self.max_new_tokens)

    def run(self, inputs):
        self.last_raw = ""
        err = ""
        try:
            return self.chain.invoke(inputs), "ok"
        except Exception as e:                                   # parsing/validation (or generation) failed
            err = str(e)
        if self.last_raw:
            try:                                                 # 1) controlled repair / retry
                return self.parser.parse(clean_json(self._repair(self.last_raw, err))), "repaired"
            except Exception:
                pass
        try:                                                     # 2) safe rule-based fallback
            return self.fallback(inputs), "fallback"
        except Exception:
            return self.model(), "fallback"


STAGE_KEYS = [("found", "Foundation"), ("core", "Core Skills"), ("adv", "Advanced Skills"), ("proj", "Projects"),
              ("deploy", "Deployment"), ("job", "Job Readiness"), ("ready", "Job Readiness"), ("interview", "Job Readiness")]


def norm_stage(s):
    s = (s or "").lower()
    for k, v in STAGE_KEYS:
        if k in s:
            return v
    return "Core Skills"


class CareerChains:
    """The six logical chains. Each one is a Runnable: state(dict) -> dict of new state entries."""

    def __init__(self, llm, store, retriever, matcher):
        self.llm, self.store, self.retriever, self.matcher = llm, store, retriever, matcher
        self.cv_llm = StructuredChain("cv_analysis", P.CV_PROMPT, llm, CVProfile, 900, self._cv_fallback)
        self.gap_llm = StructuredChain("gap_analysis", P.GAP_PROMPT, llm, GapEnrichment, 800, lambda i: GapEnrichment())
        self.road_llm = StructuredChain("roadmap", P.ROADMAP_PROMPT, llm, RoadmapOutput, 1300, self._roadmap_fallback)
        self.rep_llm = StructuredChain("final_report", P.REPORT_PROMPT, llm, Narrative, 900, self._report_fallback)
        self.chain1 = RunnableLambda(self._cv_analysis)
        self.chain2 = RunnableLambda(self._requirements)
        self.chain3 = RunnableLambda(self._skill_gap)
        self.chain4 = RunnableLambda(self._recommendation)
        self.chain5 = RunnableLambda(self._roadmap)
        self.chain6 = RunnableLambda(self._final_report)

    # ---------- helpers ----------
    @staticmethod
    def _note(state, docs, used_for):
        for d in docs:
            ev = state["evidence"].setdefault(d["id"], {"doc": d, "used_for": []})
            if used_for not in ev["used_for"]:
                ev["used_for"].append(used_for)

    # ---------- Chain 1: CV analysis ----------
    def _cv_analysis(self, state):
        profile, status = self.cv_llm.run({"cv_text": state["cv_text"][:7000]})
        state["status"]["cv_analysis"] = status
        return {"profile": profile}

    def _cv_fallback(self, inputs):
        text = inputs["cv_text"]
        ctx = {"cv_text": text, "cv_n": " " + norm(text) + " "}
        found = [name for key, name in self.store.vocab.items() if self.matcher.in_text(name, key, ctx)]
        first = next((l.strip() for l in text.splitlines() if l.strip()), "Unknown")
        return CVProfile(name=first[:60], skills=found)

    # ---------- Chain 2: career requirement analysis (RAG, no LLM call) ----------
    def _requirements(self, state):
        key, name = state["target_key"], state["target_name"]
        docs = self.retriever.requirements(key, name)
        self._note(state, docs, f"Requirements of {name}")
        return {"required": self.store.role_skills[key]}

    # ---------- Chain 3: skill-gap analysis ----------
    def _skill_gap(self, state):
        key, name, level = state["target_key"], state["target_name"], state["level"]
        matches = self.matcher.classify(state["required"], state["ctx"])
        score = self.matcher.score(matches)
        weak = [m for m in matches if m["status"] in ("Missing", "Partial Match")]
        weak.sort(key=lambda m: (-m["weight"], m["status"] != "Missing", m["skill"]))
        weak = weak[:15]
        top = weak[:6]
        docs = self.retriever.for_gaps(key, name, [m["skill"] for m in top]) if top else []
        self._note(state, docs, f"Skill-gap evidence for {name}")
        details = {}
        if top:
            lines = "\n".join(f"- {m['skill']} ({m['status']}, importance: {PRIORITY[m['weight']]})" for m in top)
            enrich, status = self.gap_llm.run({"role": name, "level": level, "missing": lines, "context": format_context(docs)})
            state["status"]["gap_analysis"] = status
            details = {norm(g.skill): g for g in enrich.gaps if g.skill}
        gaps = []
        for m in weak:
            det = details.get(m["key"])
            ev = [d["id"] for d in docs if m["key"] in norm(d["metadata"]["skill"]) or m["key"] in norm(d["text"])]
            reason = det.reason if det and det.reason else f"{m['skill']} is listed as a requirement for {name} in the knowledge base."
            gaps.append(SkillGap(skill=m["skill"], status=m["status"], priority=PRIORITY[m["weight"]], reason=reason,
                                 prerequisites=det.prerequisites if det else [], evidence_ids=ev[:3]))
        prereq_names = {norm(p) for g in gaps for p in g.prerequisites}
        gaps.sort(key=lambda g: (norm(g.skill) not in prereq_names, {"High": 0, "Medium": 1, "Low": 2}[g.priority], g.status != "Missing"))
        for i, g in enumerate(gaps, 1):
            g.learning_order = i
        sm = [SkillMatch(skill=m["skill"], status=m["status"], priority=PRIORITY[m["weight"]], matched_with=m["matched_with"]) for m in matches]
        return {"matches": matches, "skill_matches": sm, "gaps": gaps, "score": score}

    # ---------- Chain 4: career recommendation ----------
    def _recommendation(self, state):
        ranked = self.matcher.rank_roles(state["ctx"])
        recs = []
        for r in ranked[:5]:
            good = [m["skill"] for m in r["matches"] if m["status"] in ("Strong Match", "Match")]
            weak = sorted([m for m in r["matches"] if m["status"] in ("Missing", "Partial Match")], key=lambda m: -m["weight"])
            n = len(r["matches"])
            why = f"{len(good)} of {n} knowledge-base requirements are covered."
            if good:
                why += " Strongest overlap: " + ", ".join(good[:4]) + "."
            if weak:
                why += " Biggest gaps: " + ", ".join(m["skill"] for m in weak[:3]) + "."
            recs.append(RoleRecommendation(role=r["role"], score=r["score"], matched_skills=good[:8],
                                           missing_skills=[m["skill"] for m in weak[:5]], rationale=why))
        return {"ranked": ranked, "recommendations": recs}

    # ---------- Chain 5: personalised roadmap ----------
    def _roadmap(self, state):
        key, name = state["target_key"], state["target_name"]
        gaps = state["gaps"][:8]
        docs = self.retriever.for_roadmap(key, name, [g.skill for g in gaps])
        self._note(state, docs, f"Learning roadmap for {name}")
        strengths = [m["skill"] for m in state["matches"] if m["status"] in ("Strong Match", "Match")][:12]
        inputs = {
            "role": name, "level": state["level"], "strengths": ", ".join(strengths) or "none detected",
            "cv_projects": "; ".join(state["profile"].projects[:4]) or "none listed",
            "gaps": "\n".join(f"{g.learning_order}. {g.skill} ({g.priority}) | prerequisites: {', '.join(g.prerequisites) or 'none'}" for g in gaps) or "none - focus on projects and job readiness",
            "context": format_context(docs),
            "gap_list": [{"skill": g.skill, "priority": g.priority, "prerequisites": g.prerequisites, "reason": g.reason} for g in gaps],
        }
        out, status = self.road_llm.run(inputs)
        state["status"]["roadmap"] = status
        steps = [s for s in out.steps if s.skill.strip()][:9]
        for s in steps:
            s.stage = norm_stage(s.stage)
        steps.sort(key=lambda s: STAGES.index(s.stage))
        return {"roadmap": steps, "roadmap_docs": docs}

    def _roadmap_fallback(self, inputs):
        gaps, n, steps = inputs.get("gap_list", []), len(inputs.get("gap_list", [])), []
        for i, g in enumerate(gaps[:8]):
            stage = "Foundation" if i < max(1, n // 3) else ("Core Skills" if i < max(2, 2 * n // 3) else "Advanced Skills")
            steps.append(RoadmapStep(stage=stage, skill=g["skill"], why_it_matters=g["reason"], priority=g["priority"],
                                     prerequisites=g["prerequisites"],
                                     suggested_practice=f"Follow a structured tutorial on {g['skill']}, then rebuild a small exercise from scratch.",
                                     suggested_project=f"Build a small portfolio project that uses {g['skill']} in a {inputs['role']} scenario."))
        steps.append(RoadmapStep(stage="Job Readiness", skill="Portfolio and interview preparation",
                                 why_it_matters="Recruiters judge evidence of applied work.", priority="Medium",
                                 suggested_practice="Prepare concise explanations of each project.",
                                 suggested_project="Publish your best projects with clear READMEs."))
        return RoadmapOutput(steps=steps)

    # ---------- Chain 6: final career report narrative ----------
    def _final_report(self, state):
        name = state["target_name"]
        gaps = state["gaps"][:6]
        docs = self.retriever.for_gaps(state["target_key"], name, [g.skill for g in gaps[:3]], per_skill=1, cap=3) if gaps else []
        docs = (state.get("roadmap_docs") or [])[:3] + docs
        self._note(state, docs, "Project and improvement recommendations")
        strengths = [m["skill"] for m in state["matches"] if m["status"] in ("Strong Match", "Match")][:12]
        inputs = {
            "role": name, "level": state["level"], "score": state["score"],
            "strengths": ", ".join(strengths) or "none detected",
            "cv_projects": "; ".join(state["profile"].projects[:4]) or "none listed",
            "gaps": ", ".join(f"{g.skill} ({g.priority})" for g in gaps) or "no major gaps found",
            "roadmap_focus": ", ".join(s.skill for s in state["roadmap"][:6]),
            "context": format_context(docs[:6], 300),
            "gap_names": [g.skill for g in gaps], "strength_names": strengths,
        }
        out, status = self.rep_llm.run(inputs)
        state["status"]["final_report"] = status
        base = self._report_fallback(inputs)
        if not out.overall_assessment.strip():
            out.overall_assessment = base.overall_assessment
        out.projects = [p for p in out.projects if p.title.strip()][:3] or base.projects
        out.improvement_plan = out.improvement_plan[:7] or base.improvement_plan
        out.job_readiness = out.job_readiness[:5] or base.job_readiness
        return {"narrative": out}

    def _report_fallback(self, inputs):
        gaps = inputs.get("gap_names", [])
        strengths = inputs.get("strength_names", [])
        role = inputs["role"]
        return Narrative(
            overall_assessment=(f"Your profile match score for {role} is {inputs['score']}/100 (an internal matching score, not a hiring probability). "
                                + (f"You already show: {', '.join(strengths[:5])}. " if strengths else "")
                                + (f"The main gaps to close are: {', '.join(gaps[:4])}." if gaps else "No major gaps were found against the knowledge base.")),
            projects=[ProjectIdea(title=f"{role} portfolio project: {g}", description=f"Build an end-to-end project that applies {g} to a realistic {role} problem and document the results.",
                                  skills_covered=[g]) for g in gaps[:3]] or [ProjectIdea(title=f"Capstone {role} project", description="Build and deploy an end-to-end project and document it.", skills_covered=strengths[:3])],
            improvement_plan=[f"Learn {g} and apply it in a small project." for g in gaps[:4]]
                             + ["Add measurable results to your CV bullet points.", "Publish your projects on GitHub with clear READMEs.", "Practise explaining your projects in interviews."],
            job_readiness=[f"Ready: {', '.join(strengths[:4])}." if strengths else "Core strengths need to be made more visible on the CV.",
                           f"Not yet ready: {', '.join(gaps[:3])}." if gaps else "No critical gaps detected.",
                           "Strengthen your portfolio before applying widely."],
        )
''')
from career_coach.matching import SkillMatcher
from career_coach.chains import CareerChains, StructuredChain, clean_json

print("✅ Chains built: 1 CV analysis | 2 Requirement analysis | 3 Skill gap | 4 Recommendation | 5 Roadmap | 6 Final report")

---
## Step 12 · Career Coach pipeline (+ local PDF CV parser)

Writes the `cv_parser` and `pipeline` modules and creates the `CareerCoach` object.

In [ ]:
# CELL 12 — Career Coach pipeline (+ local PDF CV parser)
write_module("cv_parser", r'''
import io
import re


class CVError(Exception):
    """User-facing problem with the uploaded CV."""


def extract_cv_text(data, max_chars=12000):
    """Local PDF text extraction (PyMuPDF, with pypdf as backup). Nothing leaves the machine."""
    if not data or not data[:5].startswith(b"%PDF"):
        raise CVError("This file does not look like a valid PDF. Please upload your CV as a PDF.")
    text = ""
    try:
        import fitz
        with fitz.open(stream=data, filetype="pdf") as doc:
            if getattr(doc, "needs_pass", False):
                raise CVError("This PDF is password-protected. Please upload an unprotected copy.")
            text = "\n".join(page.get_text("text") for page in doc)
    except CVError:
        raise
    except Exception:
        text = ""
    if len(text.strip()) < 30:
        try:
            from pypdf import PdfReader
            text = "\n".join((p.extract_text() or "") for p in PdfReader(io.BytesIO(data)).pages)
        except Exception:
            text = ""
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    if len(text) < 80:
        raise CVError("No readable text could be extracted. The PDF may be a scan or an image - please upload a text-based PDF.")
    return text[:max_chars]
''')

write_module("pipeline", r'''
import hashlib
import re

from .chains import CareerChains
from .matching import SkillMatcher
from .retriever import CareerRetriever
from .schemas import *

SCORE_NOTE = ("Profile Match Score = internal matching between the skills found in your CV and the role requirements in the "
              "knowledge base. It is NOT a real-world hiring probability.")


class CoachError(Exception):
    """User-facing pipeline problem."""


def experience_level(profile):
    y = profile.years_of_experience
    if y >= 6:
        return "Senior"
    if y >= 3:
        return "Mid"
    if y >= 1:
        return "Junior"
    s = profile.experience_level.lower()
    for key, label in (("senior", "Senior"), ("mid", "Mid"), ("junior", "Junior"), ("student", "Student"), ("entry", "Entry")):
        if key in s:
            return label
    return "Entry"


class CareerCoach:
    def __init__(self, llm, store):
        self.llm, self.store = llm, store
        self.retriever = CareerRetriever(store)
        self.matcher = SkillMatcher(store)
        self.chains = CareerChains(llm, store, self.retriever, self.matcher)
        self._cache = {}

    def role_names(self):
        return sorted(self.store.role_names[k] for k in self.store.rankable_roles())

    def analyze(self, cv_text, target_role=None, extra_skills="", progress=None):
        if not cv_text or len(cv_text.strip()) < 80:
            raise CoachError("The CV text is empty or too short to analyse.")
        if not self.store.rankable_roles():
            raise CoachError("The knowledge base has no roles with skill data ('role' + 'skill' metadata), so no career matching is possible.")
        cache_key = hashlib.sha1((cv_text + "|" + str(target_role) + "|" + str(extra_skills)).encode("utf-8")).hexdigest()
        if cache_key in self._cache:
            return self._cache[cache_key]

        def step(frac, msg):
            if progress:
                progress(frac, msg)

        c = self.chains
        state = {"cv_text": cv_text.strip(), "status": {}, "evidence": {}, "warnings": []}

        step(0.05, "Chain 1/6 · Reading your CV with the local LLM…")
        state.update(c.chain1.invoke(state))
        profile = state["profile"]
        state["level"] = experience_level(profile)
        extra = [x.strip() for x in re.split(r"[,;\n]", extra_skills or "") if x.strip()]
        state["user_skills"] = list(dict.fromkeys(profile.all_skills() + extra))
        state["ctx"] = self.matcher.prepare(state["user_skills"], state["cv_text"])

        step(0.30, "Chain 4/6 · Ranking career paths against the knowledge base…")
        state.update(c.chain4.invoke(state))

        target_key, auto = None, True
        if target_role and target_role.strip():
            target_key = self.store.resolve_role(target_role)
            if target_key and target_key in self.store.rankable_roles():
                auto = False
            else:
                target_key = None
                state["warnings"].append(f"The role '{target_role}' was not found in the knowledge base, so the best-matching career was analysed instead.")
        if target_key is None:
            target_key = state["ranked"][0]["key"]
        state["target_key"], state["target_name"] = target_key, self.store.role_names[target_key]

        step(0.40, f"Chain 2/6 · Retrieving requirements for {state['target_name']}…")
        state.update(c.chain2.invoke(state))
        step(0.50, "Chain 3/6 · Analysing skill gaps…")
        state.update(c.chain3.invoke(state))
        step(0.70, "Chain 5/6 · Building your personalised roadmap…")
        state.update(c.chain5.invoke(state))
        step(0.88, "Chain 6/6 · Writing the final career report…")
        state.update(c.chain6.invoke(state))

        if any(v == "fallback" for v in state["status"].values()):
            state["warnings"].append("Part of the analysis used a rule-based fallback because the local model's output could not be parsed. Results are still based on the knowledge base.")

        evidence = []
        for rid, ev in state["evidence"].items():
            d = ev["doc"]
            m = d["metadata"]
            evidence.append(EvidenceItem(id=rid, role=m["role"], skill=m["skill"], category=m["category"], source_type=m["source_type"],
                                         snippet=d["text"][:450], used_for="; ".join(ev["used_for"])))
        matches = state["matches"]
        n = state["narrative"]
        report = CareerReport(
            target_role=state["target_name"], auto_recommended=auto, profile_match_score=state["score"], score_note=SCORE_NOTE,
            experience_level=state["level"], overall_assessment=n.overall_assessment, profile=profile,
            current_skills=state["user_skills"],
            strong_skills=[m["skill"] for m in matches if m["status"] in ("Strong Match", "Match")],
            partial_skills=[m["skill"] for m in matches if m["status"] == "Partial Match"],
            missing_skills=[m["skill"] for m in matches if m["status"] == "Missing"],
            skill_matches=state["skill_matches"], skill_gaps=state["gaps"], recommendations=state["recommendations"],
            roadmap=state["roadmap"], projects=n.projects, improvement_plan=n.improvement_plan, job_readiness=n.job_readiness,
            evidence=evidence, pipeline_status=state["status"], warnings=state["warnings"],
        )
        step(1.0, "Done")
        self._cache[cache_key] = report
        return report
''')
from career_coach.cv_parser import extract_cv_text, CVError
from career_coach.pipeline import CareerCoach, CoachError

coach = CareerCoach(llm, store) if (llm and store) else None
print("✅ CareerCoach pipeline ready" if coach else "⏭️  Pipeline not created (missing knowledge base or LLM).")

---
## Step 13 · Automated tests (no external API needed)

Runs 9 automated checks, then frees GPU memory for Streamlit.

In [ ]:
# CELL 13 — Automated tests (no external API needed)
SAMPLE_CV = """Alex Morgan
Junior Data Analyst | alex.morgan@example.com
EDUCATION
BSc Computer Science, State University, 2022
EXPERIENCE
Data Analyst - Retail Corp (2022-2024): built SQL reports and Excel dashboards, automated data cleaning with Python and pandas,
trained a simple scikit-learn churn model, used Git for version control.
PROJECTS
Customer churn prediction (Python, pandas, scikit-learn). Sales dashboard (SQL, Power BI).
SKILLS
Python, SQL, pandas, NumPy, scikit-learn, Git, Excel, Power BI, communication, teamwork
CERTIFICATIONS
Google Data Analytics Certificate"""

results = []
def check(name, fn):
    try:
        info = fn()
        results.append((name, True))
        print(f"✅ {name}" + (f" — {info}" if info else ""))
    except Exception as e:
        results.append((name, False))
        print(f"❌ {name} — {type(e).__name__}: {e}")

def t_rag():
    assert RECORDS, RAG_ERROR or "RAG file not loaded"
    return RAG_PATH
def t_count():
    assert RECORDS
    return f"{len(RECORDS)} records"
def t_embed():
    v = embedder.embed_documents(["python", "sql"])
    assert v.shape == (2, embedder.dim) and abs(float((v[0] ** 2).sum()) - 1) < 1e-2
    return f"dim={embedder.dim}"
def t_faiss():
    assert store and store.index.ntotal == len(RECORDS)
    return f"{store.index.ntotal} vectors"
def t_retrieve():
    rk = store.rankable_roles()[0]
    name = store.role_names[rk]
    docs = retriever.requirements(rk, name)
    assert docs, "no documents returned"
    return f"{len(docs)} docs for '{name}', top score {docs[0]['score']:.2f}"
def t_llm():
    assert llm, LLM_ERROR
    out = llm.generate([{"role": "user", "content": "Reply with the single word: OK"}], 8)
    return f"{llm.model_name} -> {out.strip()[:20]!r}"
def t_cv_schema():
    p = CVProfile.model_validate({"name": "A", "skills": "Python, SQL", "years_of_experience": "2 years", "projects": None})
    assert p.years_of_experience == 2.0 and p.skills == ["Python, SQL"] and "Python" in p.all_skills()
def t_report_schema():
    r = CareerReport(target_role="X", skill_gaps=[SkillGap(skill="Docker")], roadmap=[RoadmapStep(skill="Docker")])
    CareerReport.model_validate(r.model_dump())
REPORT = None
def t_pipeline():
    global REPORT
    assert coach, "pipeline unavailable (needs knowledge base + LLM)"
    t0 = time.time()
    REPORT = coach.analyze(SAMPLE_CV, target_role=None)
    CareerReport.model_validate(REPORT.model_dump())
    assert REPORT.target_role and REPORT.roadmap and REPORT.evidence
    return (f"{time.time()-t0:.0f}s | auto target='{REPORT.target_role}' score={REPORT.profile_match_score} | "
            f"gaps={len(REPORT.skill_gaps)} roadmap={len(REPORT.roadmap)} evidence={len(REPORT.evidence)} | chains={REPORT.pipeline_status}")

print("=" * 70)
check("1. RAG file loaded", t_rag)
check("2. Number of records", t_count)
check("3. Embeddings work", t_embed)
check("4. FAISS index works", t_faiss)
check("5. Retrieval returns relevant documents", t_retrieve)
check("6. Local LLM loads and answers", t_llm)
check("7. CVProfile schema validates", t_cv_schema)
check("8. CareerReport schema validates", t_report_schema)
check("9. Complete pipeline on a sample CV", t_pipeline)
print("=" * 70)
print(f"Tests passed: {sum(ok for _, ok in results)}/{len(results)}")
if REPORT:
    print("\nTop recommendations (Profile Match Score):")
    for i, r in enumerate(REPORT.recommendations[:3], 1):
        print(f"  {i}. {r.role} — {r.score}")

# Free GPU memory: the Streamlit app loads its own copy (from the local disk cache) in a separate process.
for _n in ("coach", "llm"):
    globals().pop(_n, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\n🧹 Notebook GPU memory released — Streamlit will load the models itself.")

---
## Step 14 · Create the Streamlit application (app.py + theme)

Writes the Streamlit theme and `app.py`.

In [ ]:
# CELL 14 — Create the Streamlit application (app.py + theme)
(WORK / ".streamlit").mkdir(exist_ok=True)
(WORK / ".streamlit" / "config.toml").write_text('''[theme]
base = "light"
primaryColor = "#1F4E79"
backgroundColor = "#FFFFFF"
secondaryBackgroundColor = "#F5F7FA"
textColor = "#1B2430"
font = "sans serif"

[server]
headless = true
enableCORS = false
enableXsrfProtection = false

[browser]
gatherUsageStats = false
''')

APP_CODE = r'''
import html
import sys
import traceback
from pathlib import Path

import streamlit as st

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
sys.path.insert(0, str(WORK))

from career_coach.cv_parser import CVError, extract_cv_text
from career_coach.rag_loader import RAGError, find_rag_file, load_records

st.set_page_config(page_title="AI Career Coach", page_icon="🧭", layout="wide")

st.markdown("""
<style>
#MainMenu, footer {visibility: hidden;}
.block-container {padding-top: 2rem; max-width: 1180px;}
.hero h1 {font-size: 2.1rem; margin: 0; color: #12263A; letter-spacing: -0.5px;}
.hero p {color: #5B6675; margin: .2rem 0 1.2rem 0; font-size: 1.02rem;}
.card {background: #fff; border: 1px solid #E3E8EF; border-radius: 12px; padding: 16px 18px; margin-bottom: 12px;}
.card h4 {margin: 0 0 6px 0; color: #12263A; font-size: 1.02rem;}
.muted {color: #5B6675; font-size: .88rem;}
.chip {display: inline-block; padding: 3px 11px; border-radius: 999px; font-size: .8rem; margin: 2px 5px 2px 0; border: 1px solid transparent;}
.chip-good {background: #E8F5EE; color: #17633A; border-color: #BFE3CD;}
.chip-ok {background: #E8F0FB; color: #1F4E8C; border-color: #C5D8F3;}
.chip-part {background: #FFF4E0; color: #8A5A00; border-color: #F3DDB0;}
.chip-miss {background: #FDECEC; color: #9B2C2C; border-color: #F4C7C7;}
.chip-neutral {background: #F1F3F7; color: #34404F; border-color: #E0E5EC;}
.bar {height: 8px; background: #EDF0F5; border-radius: 6px; overflow: hidden; margin: 6px 0 4px 0;}
.bar > div {height: 100%; background: #1F4E79; border-radius: 6px;}
.kpi {background: #F5F7FA; border: 1px solid #E3E8EF; border-radius: 12px; padding: 14px 16px;}
.kpi .l {color: #5B6675; font-size: .78rem; text-transform: uppercase; letter-spacing: .6px;}
.kpi .v {color: #12263A; font-size: 1.6rem; font-weight: 650; line-height: 1.25;}
.stage {color: #1F4E79; font-weight: 650; font-size: .82rem; text-transform: uppercase; letter-spacing: .8px; margin: 18px 0 6px 0;}
</style>
""", unsafe_allow_html=True)

AUTO = "✨ Auto-recommend my best career"


def esc(x):
    return html.escape(str(x))


def chips(items, kind="neutral"):
    return "".join(f'<span class="chip chip-{kind}">{esc(i)}</span>' for i in items if str(i).strip()) or '<span class="muted">—</span>'


def log_exc(e):
    try:
        with open(WORK / "app_errors.log", "a") as fh:
            fh.write(traceback.format_exc() + "\n")
    except Exception:
        pass


@st.cache_resource(show_spinner="Loading knowledge base, local embedding model and local LLM — the first start takes 1–3 minutes…")
def load_system():
    from career_coach.embeddings import LocalEmbedder
    from career_coach.llm import LocalLLM
    from career_coach.pipeline import CareerCoach
    from career_coach.vectorstore import FaissStore

    path = find_rag_file()
    records = load_records(path)
    embedder = LocalEmbedder()
    store = FaissStore(records, embedder)
    llm = LocalLLM()
    return {"coach": CareerCoach(llm, store), "records": len(records), "llm": llm.model_name, "device": llm.device}


# ------------------------------------------------------------------ header
st.markdown('<div class="hero"><h1>🧭 AI Career Coach</h1>'
            '<p>Upload your CV and get a skill-gap analysis, career recommendations and a personalised roadmap — '
            'powered by a local open-source LLM and a retrieval knowledge base. Your CV never leaves this machine.</p></div>',
            unsafe_allow_html=True)

system, load_error = None, None
try:
    system = load_system()
except RAGError as e:
    load_error = str(e)
except Exception as e:
    log_exc(e)
    load_error = "The AI models could not be loaded. Make sure the notebook ran without errors (GPU enabled), then press Reload."

# ------------------------------------------------------------------ sidebar
with st.sidebar:
    st.markdown("### Your profile")
    uploaded = st.file_uploader("Upload your CV (PDF)", type=["pdf"])
    roles = system["coach"].role_names() if system else []
    choice = st.selectbox("Target career", [AUTO] + roles, help="Pick a role to analyse your fit, or let the coach recommend one.")
    extra = st.text_area("Additional skills (optional)", placeholder="e.g. Docker, Tableau, public speaking", height=80)
    run = st.button("Analyze my CV", type="primary", use_container_width=True, disabled=system is None)
    if system:
        st.caption(f"Knowledge base: {system['records']} entries · {len(roles)} roles\n\nModel: {system['llm']} ({system['device']})")
    if load_error and st.button("Reload", use_container_width=True):
        st.cache_resource.clear()
        st.rerun()

if load_error:
    st.error(load_error)

# ------------------------------------------------------------------ analysis
if run and system:
    if uploaded is None:
        st.warning("Please upload your CV as a PDF first.")
    else:
        try:
            cv_text = extract_cv_text(uploaded.getvalue())
            with st.status("Analysing your CV…", expanded=True) as status:
                bar = st.progress(0.0, text="Starting…")

                def cb(frac, msg):
                    bar.progress(min(max(frac, 0.0), 1.0), text=msg)
                    status.update(label=msg)

                report = system["coach"].analyze(cv_text, None if choice == AUTO else choice, extra, cb)
                status.update(label="Analysis complete", state="complete", expanded=False)
            st.session_state["report"] = report
        except (CVError,) as e:
            st.error(str(e))
        except Exception as e:
            log_exc(e)
            msg = str(e) if e.__class__.__name__ == "CoachError" else "Something went wrong while analysing your CV. Please try again, or try a different PDF."
            st.error(msg)

report = st.session_state.get("report")

# ------------------------------------------------------------------ landing
if report is None:
    if system and not load_error:
        c1, c2, c3 = st.columns(3)
        for col, (t, d) in zip((c1, c2, c3), [
            ("1 · Upload", "Add your CV (PDF) in the sidebar. Text is extracted locally."),
            ("2 · Choose", "Select a target career or let the coach recommend the best match."),
            ("3 · Analyse", "Six chains run: CV analysis, requirements, skill gaps, recommendations, roadmap, report.")]):
            col.markdown(f'<div class="card"><h4>{t}</h4><span class="muted">{d}</span></div>', unsafe_allow_html=True)
    st.stop()

# ------------------------------------------------------------------ results
for w in report.warnings:
    st.warning(w)

score = report.profile_match_score
label = "Strong profile match" if score >= 75 else "Promising match" if score >= 55 else "Developing match" if score >= 35 else "Early-stage match"
tabs = st.tabs(["Overview", "Current Skills", "Skill Gap", "Career Paths", "Roadmap", "Projects", "Improvement Plan", "Evidence"])

with tabs[0]:
    st.markdown("#### Career Overview")
    if report.auto_recommended:
        st.info("No target career was selected, so the best-matching career in the knowledge base was analysed.")
    k1, k2, k3, k4 = st.columns(4)
    for col, (l, v) in zip((k1, k2, k3, k4), [("Target role", report.target_role), ("Profile Match Score", f"{score:.0f} / 100"),
                                               ("Experience level", report.experience_level), ("Skill gaps", str(len(report.skill_gaps)))]):
        col.markdown(f'<div class="kpi"><div class="l">{l}</div><div class="v">{esc(v)}</div></div>', unsafe_allow_html=True)
    st.markdown(f'<div class="bar"><div style="width:{min(score, 100)}%"></div></div><span class="muted">{label} · {esc(report.score_note)}</span>', unsafe_allow_html=True)
    st.markdown(f'<div class="card" style="margin-top:14px"><h4>Overall assessment</h4>{esc(report.overall_assessment)}</div>', unsafe_allow_html=True)
    p = report.profile
    a, b = st.columns(2)
    a.markdown(f'<div class="card"><h4>Candidate</h4><b>{esc(p.name)}</b><br><span class="muted">Education</span><br>{"<br>".join(esc(x) for x in p.education) or "—"}</div>', unsafe_allow_html=True)
    b.markdown(f'<div class="card"><h4>Experience</h4>{"<br>".join(esc(x) for x in p.experience) or "—"}<br><span class="muted">Certifications</span><br>{"<br>".join(esc(x) for x in p.certifications) or "—"}</div>', unsafe_allow_html=True)
    st.download_button("Download full report (JSON)", report.model_dump_json(indent=2), file_name="career_report.json", mime="application/json")

with tabs[1]:
    st.markdown("#### Current Skills")
    p = report.profile
    groups = [("Programming languages", p.programming_languages), ("Frameworks", p.frameworks), ("Libraries", p.libraries),
              ("Databases", p.databases), ("Cloud technologies", p.cloud_technologies), ("Tools", p.tools), ("Soft skills", p.soft_skills)]
    st.markdown(f'<div class="card"><h4>Strong for {esc(report.target_role)}</h4>{chips(report.strong_skills, "good")}</div>', unsafe_allow_html=True)
    cols = st.columns(2)
    shown = [(t, v) for t, v in groups if v]
    for i, (t, v) in enumerate(shown):
        cols[i % 2].markdown(f'<div class="card"><h4>{t}</h4>{chips(v, "ok" if t != "Soft skills" else "neutral")}</div>', unsafe_allow_html=True)
    if not shown:
        st.markdown(f'<div class="card"><h4>All detected skills</h4>{chips(report.current_skills, "ok")}</div>', unsafe_allow_html=True)

with tabs[2]:
    st.markdown(f"#### Skill Gap — {esc(report.target_role)}")
    counts = {s: sum(1 for m in report.skill_matches if m.status == s) for s in ("Strong Match", "Match", "Partial Match", "Missing")}
    for col, (s, n) in zip(st.columns(4), counts.items()):
        col.metric(s, n)
    sm_chips = ""
    for m in report.skill_matches:
        kind = {"Strong Match": "good", "Match": "ok", "Partial Match": "part", "Missing": "miss"}[m.status] if m.status in ("Strong Match", "Match", "Partial Match", "Missing") else "neutral"
        sm_chips += f'<span class="chip chip-{kind}" title="{esc(m.status)}">{esc(m.skill)}</span>'
    st.markdown(f'<div class="card"><h4>Requirement coverage</h4>{sm_chips}<div class="muted" style="margin-top:6px">Green = strong match · blue = match · amber = partial · red = missing</div></div>', unsafe_allow_html=True)
    if not report.skill_gaps:
        st.success("No gaps found against the knowledge-base requirements for this role.")
    for g in report.skill_gaps:
        pk = {"High": "miss", "Medium": "part", "Low": "ok"}.get(g.priority, "neutral")
        st.markdown(
            f'<div class="card"><h4>{g.learning_order}. {esc(g.skill)} <span class="chip chip-{pk}">{esc(g.priority)} priority</span>'
            f'<span class="chip chip-neutral">{esc(g.status)}</span></h4>{esc(g.reason)}'
            f'<div class="muted" style="margin-top:6px">Prerequisites: {esc(", ".join(g.prerequisites) or "none")}'
            f'{" · Evidence: " + esc(", ".join(g.evidence_ids)) if g.evidence_ids else ""}</div></div>', unsafe_allow_html=True)

with tabs[3]:
    st.markdown("#### Recommended Career Paths")
    st.caption("Ranked by Profile Match Score — an internal matching score against the knowledge base, not a hiring probability.")
    for i, r in enumerate(report.recommendations, 1):
        st.markdown(
            f'<div class="card"><h4>{i}. {esc(r.role)} <span class="chip chip-ok">Profile Match Score: {r.score:.0f}</span></h4>'
            f'<div class="bar"><div style="width:{min(r.score, 100)}%"></div></div>{esc(r.rationale)}'
            f'<div style="margin-top:6px">{chips(r.matched_skills[:6], "good")}{chips(r.missing_skills[:4], "miss")}</div></div>', unsafe_allow_html=True)

with tabs[4]:
    st.markdown("#### Personalised Roadmap")
    if not report.roadmap:
        st.info("No roadmap steps were generated.")
    for stage in ["Foundation", "Core Skills", "Advanced Skills", "Projects", "Deployment", "Job Readiness"]:
        steps = [s for s in report.roadmap if s.stage == stage]
        if not steps:
            continue
        st.markdown(f'<div class="stage">{stage}</div>', unsafe_allow_html=True)
        for s in steps:
            pk = {"High": "miss", "Medium": "part", "Low": "ok"}.get(s.priority, "neutral")
            st.markdown(
                f'<div class="card"><h4>{esc(s.skill)} <span class="chip chip-{pk}">{esc(s.priority)}</span></h4>{esc(s.why_it_matters)}'
                f'<div class="muted" style="margin-top:8px"><b>Prerequisites:</b> {esc(", ".join(s.prerequisites) or "none")}<br>'
                f'<b>Practice:</b> {esc(s.suggested_practice)}<br><b>Project:</b> {esc(s.suggested_project)}</div></div>', unsafe_allow_html=True)

with tabs[5]:
    st.markdown("#### Recommended Projects")
    for pr in report.projects:
        st.markdown(f'<div class="card"><h4>{esc(pr.title)}</h4>{esc(pr.description)}<div style="margin-top:6px">{chips(pr.skills_covered, "ok")}</div></div>', unsafe_allow_html=True)

with tabs[6]:
    st.markdown("#### Improvement Plan")
    for i, step in enumerate(report.improvement_plan, 1):
        st.markdown(f'<div class="card"><b>{i}.</b> {esc(step)}</div>', unsafe_allow_html=True)
    st.markdown("#### Job-readiness guidance")
    for line in report.job_readiness:
        st.markdown(f"- {line}")

with tabs[7]:
    st.markdown("#### Evidence")
    st.caption("Entries below were retrieved from the knowledge base (RAG) and given to the model. The roadmap, projects and plan are "
               "AI-generated recommendations written from this evidence and your CV — they are not quoted sources or statistics.")
    st.markdown(f"**{len(report.evidence)} knowledge-base entries used** · pipeline status: `{report.pipeline_status}`")
    for ev in report.evidence:
        with st.expander(f"{ev.id} · {ev.role or 'general'} · {ev.skill or ev.category or 'entry'}"):
            st.markdown(f"**Used for:** {ev.used_for}")
            st.markdown(f"**Category:** {ev.category or '—'} · **Source type:** {ev.source_type or '—'}")
            st.write(ev.snippet)
'''
(WORK / "app.py").write_text(APP_CODE, encoding="utf-8")
print("✅ Streamlit app written to", WORK / "app.py")

---
## Final step · Launch Streamlit + free Cloudflare tunnel (no account, no API key). Keep this cell running.

Starts Streamlit and a free tunnel, then prints the public URL. **Keep this cell running.**

In [ ]:
# FINAL CELL — Launch Streamlit + free Cloudflare tunnel (no account, no API key). Keep this cell running.
import subprocess, sys, os, time, re, socket, urllib.request
from pathlib import Path
from IPython.display import display, HTML

PORT = 8501
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

def tail(path, n=30):
    try:
        return "\n".join(Path(path).read_text(errors="ignore").splitlines()[-n:])
    except Exception:
        return "(no log)"

subprocess.run('pkill -f "[s]treamlit run"; pkill -f "[c]loudflared"', shell=True)
time.sleep(1)

env = os.environ.copy()
env["PYTHONPATH"] = f"{WORK}:{env.get('PYTHONPATH', '')}"
streamlit_log = open(WORK / "streamlit.log", "w")
streamlit_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", str(WORK / "app.py"), "--server.port", str(PORT), "--server.address", "0.0.0.0",
     "--server.headless", "true", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
     "--browser.gatherUsageStats", "false"],
    cwd=str(WORK), env=env, stdout=streamlit_log, stderr=subprocess.STDOUT)

ready = False
for _ in range(90):
    if streamlit_proc.poll() is not None:
        break
    try:
        socket.create_connection(("127.0.0.1", PORT), timeout=1).close()
        ready = True
        break
    except OSError:
        time.sleep(1)
if not ready:
    print("❌ Streamlit did not start. Log:\n" + tail(WORK / "streamlit.log"))
    raise RuntimeError("Streamlit failed to start (see log above).")
print("✅ Streamlit is running on port", PORT)

def start_cloudflared():
    binp = WORK / "cloudflared"
    if not binp.exists():
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", str(binp))
    binp.chmod(0o755)
    log = open(WORK / "tunnel.log", "w")
    proc = subprocess.Popen([str(binp), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate", "--protocol", "http2"],
                            stdout=log, stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", (WORK / "tunnel.log").read_text(errors="ignore"))
        if m:
            return proc, m.group(0), ""
        if proc.poll() is not None:
            break
    proc.terminate()
    raise RuntimeError("cloudflared returned no URL:\n" + tail(WORK / "tunnel.log", 10))

def start_localtunnel():
    log = open(WORK / "lt.log", "w")
    proc = subprocess.Popen(["npx", "--yes", "localtunnel", "--port", str(PORT)], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(90):
        time.sleep(1)
        m = re.search(r"https://[a-z0-9-]+\.loca\.lt", (WORK / "lt.log").read_text(errors="ignore"))
        if m:
            try:
                pw = urllib.request.urlopen("https://loca.lt/mytunnelpassword", timeout=10).read().decode().strip()
            except Exception:
                pw = "(open https://loca.lt/mytunnelpassword to see it)"
            return proc, m.group(0), f"If the page asks for a tunnel password, enter: {pw}"
        if proc.poll() is not None:
            break
    proc.terminate()
    raise RuntimeError("localtunnel returned no URL:\n" + tail(WORK / "lt.log", 10))

tunnel_proc, public_url, note = None, None, ""
for starter in (start_cloudflared, start_localtunnel):
    try:
        tunnel_proc, public_url, note = starter()
        break
    except Exception as e:
        print(f"⚠️  {starter.__name__} failed: {str(e)[:300]}")

if not public_url:
    print("❌ No tunnel could be created. Make sure Internet is ON in the notebook settings (right sidebar) and run this cell again.")
    streamlit_proc.terminate()
    raise RuntimeError("Tunnel unavailable.")

display(HTML(f"""
<div style="font-family:sans-serif;border:1px solid #CBD5E1;border-radius:12px;padding:18px 22px;background:#F8FAFC;max-width:760px">
  <div style="font-size:18px;font-weight:650;color:#12263A">🧭 AI Career Coach is live</div>
  <div style="margin:10px 0"><a href="{public_url}" target="_blank" style="font-size:17px;color:#1F4E79">{public_url}</a></div>
  <div style="color:#5B6675;font-size:13px">The first page load downloads nothing but loads the models (1–3 min). If the link shows an error for a few seconds, refresh. {note}<br>
  Keep this cell running — interrupt it to shut the app down.</div>
</div>"""))
print("🌐 PUBLIC URL:", public_url)

try:
    while streamlit_proc.poll() is None:
        time.sleep(5)
    print("Streamlit stopped. Log:\n" + tail(WORK / "streamlit.log"))
except KeyboardInterrupt:
    print("Shutting down…")
finally:
    for p in (tunnel_proc, streamlit_proc):
        try:
            p.terminate()
        except Exception:
            pass